# Iterative Imputation, Step by Step (MICE)

This notebook hand-builds the idea behind **Multivariate Imputation by Chained Equations (MICE)**, one tiny step at a time: start with a crude mean-imputed guess for every missing value, then repeatedly re-predict each missing value using a regression model trained on the *other* columns — cycling through columns, round after round, until the imputed values stop changing much. Seeing it done by hand first makes `IterativeImputer` (the one-line sklearn version, at the end) far less of a black box.

## 1. A Tiny, Toy Dataset

Just 5 rows and 3 numeric columns — small enough to inspect by hand at every step.

In [1]:
import pandas as pd
import numpy as np

from sklearn.linear_model import LinearRegression

In [2]:
df = np.round(pd.read_csv('50_Startups.csv')[['R&D Spend','Administration','Marketing Spend','Profit']]/10000)
np.random.seed(9)
df = df.sample(5)
df

FileNotFoundError: [Errno 2] No such file or directory: '50_Startups.csv'

In [ ]:
df = df.iloc[:,0:-1]
df

Manually injecting one missing value into each of the three columns, in different rows:

In [ ]:
df.iloc[1,0] = np.NaN
df.iloc[3,1] = np.NaN
df.iloc[-1,-1] = np.NaN

In [ ]:
df.head()

## 2. Round 0 — The Crude Starting Guess

Before any clever prediction, every missing value just gets the **mean** of its own column — a deliberately simple starting point that the iterations below will progressively refine.

In [ ]:
# Step 1 - Impute all missing values with mean of respective col

df0 = pd.DataFrame()

df0['R&D Spend'] = df['R&D Spend'].fillna(df['R&D Spend'].mean())
df0['Administration'] = df['Administration'].fillna(df['Administration'].mean())
df0['Marketing Spend'] = df['Marketing Spend'].fillna(df['Marketing Spend'].mean())

In [ ]:
# 0th Iteration
df0

## 3. Iteration 1

For each column that originally had a missing value, in turn: remove that column's **mean-imputed** value again (treating it as still-unknown), train a `LinearRegression` on the *other* two columns using the rows that have real values, and predict a better replacement.

### Column 1 (`R&D Spend`)

In [ ]:
# Remove the col1 imputed value
df1 = df0.copy()

df1.iloc[1,0] = np.NaN

df1

In [ ]:
# Use first 3 rows to build a model and use the last for prediction

X = df1.iloc[[0,2,3,4],1:3]
X

In [ ]:
y = df1.iloc[[0,2,3,4],0]
y

In [ ]:
lr = LinearRegression()
lr.fit(X,y)
lr.predict(df1.iloc[1,1:].values.reshape(1,2))

Plugging the prediction back in:

In [ ]:
df1.iloc[1,0] = 23.14

In [ ]:
df1

### Column 2 (`Administration`) — Same Process, Now Using the *Updated* Column 1

In [ ]:
# Remove the col2 imputed value

df1.iloc[3,1] = np.NaN

df1

In [ ]:
# Use last 3 rows to build a model and use the first for prediction
X = df1.iloc[[0,1,2,4],[0,2]]
X

In [ ]:
y = df1.iloc[[0,1,2,4],1]
y

In [ ]:
lr = LinearRegression()
lr.fit(X,y)
lr.predict(df1.iloc[3,[0,2]].values.reshape(1,2))

In [ ]:
df1.iloc[3,1] = 11.06

In [ ]:
df1

### Column 3 (`Marketing Spend`) — Same Process Again

In [ ]:
# Remove the col3 imputed value
df1.iloc[4,-1] = np.NaN

df1

In [ ]:
# Use last 3 rows to build a model and use the first for prediction
X = df1.iloc[0:4,0:2]
X

In [ ]:
y = df1.iloc[0:4,-1]
y

In [ ]:
lr = LinearRegression()
lr.fit(X,y)
lr.predict(df1.iloc[4,0:2].values.reshape(1,2))

In [ ]:
df1.iloc[4,-1] = 31.56

### Checking How Much Changed

Subtracting the previous round's values from this round's shows exactly how much each prediction moved — the key number to watch as more iterations run. MICE stops once these differences become negligibly small.

In [ ]:
# After 1st Iteration
df1

## 4. Iteration 2 — Repeat the Exact Same Process

Same three steps again, but now using **Iteration 1's** values as the "known" data for the other columns, instead of the original crude mean-imputed guesses.

In [ ]:
# Subtract 0th iteration from 1st iteration

df1 - df0

In [ ]:
df2 = df1.copy()

df2.iloc[1,0] = np.NaN

df2

In [ ]:
X = df2.iloc[[0,2,3,4],1:3]
y = df2.iloc[[0,2,3,4],0]

lr = LinearRegression()
lr.fit(X,y)
lr.predict(df2.iloc[1,1:].values.reshape(1,2))

In [ ]:
df2.iloc[1,0] = 23.78

In [ ]:
df2.iloc[3,1] = np.NaN
X = df2.iloc[[0,1,2,4],[0,2]]
y = df2.iloc[[0,1,2,4],1]

lr = LinearRegression()
lr.fit(X,y)
lr.predict(df2.iloc[3,[0,2]].values.reshape(1,2))

In [ ]:
df2.iloc[3,1] = 11.22

In [ ]:
df2.iloc[4,-1] = np.NaN

X = df2.iloc[0:4,0:2]
y = df2.iloc[0:4,-1]

lr = LinearRegression()
lr.fit(X,y)
lr.predict(df2.iloc[4,0:2].values.reshape(1,2))

In [ ]:
df2.iloc[4,-1] = 31.56

The changes from Iteration 1 → 2 should already be noticeably smaller than 0 → 1 — a sign of convergence starting to kick in.

In [ ]:
df2

In [ ]:
df2 - df1

## 5. Iteration 3 — One More Round

In [ ]:
df3 = df2.copy()

df3.iloc[1,0] = np.NaN

df3

In [ ]:
X = df3.iloc[[0,2,3,4],1:3]
y = df3.iloc[[0,2,3,4],0]

lr = LinearRegression()
lr.fit(X,y)
lr.predict(df3.iloc[1,1:].values.reshape(1,2))

In [ ]:
df3.iloc[1,0] = 24.57

In [ ]:
df3.iloc[3,1] = np.NaN
X = df3.iloc[[0,1,2,4],[0,2]]
y = df3.iloc[[0,1,2,4],1]

lr = LinearRegression()
lr.fit(X,y)
lr.predict(df3.iloc[3,[0,2]].values.reshape(1,2))

In [ ]:
df3.iloc[3,1] = 11.37

In [ ]:
df3.iloc[4,-1] = np.NaN

X = df3.iloc[0:4,0:2]
y = df3.iloc[0:4,-1]

lr = LinearRegression()
lr.fit(X,y)
lr.predict(df3.iloc[4,0:2].values.reshape(1,2))

In [ ]:
df3.iloc[4,-1] = 45.53

*(Re-confirming a value from Iteration 2 that this line resets for the final comparison below — redundant but harmless, since it restores the same value already set in cell 30.)*

In [ ]:
df2.iloc[3,1] = 11.22

In [ ]:
df3

The differences here should be smaller still — each round's predictions moving less than the round before is exactly what convergence looks like.

In [ ]:
df3 - df2

## 6. The One-Line Version — `IterativeImputer`

Everything above — mean-start, then repeatedly re-predicting each column from the others, round after round — is exactly what scikit-learn's `IterativeImputer` automates. It's still marked **experimental**, so it needs an explicit opt-in import before use.

In [ ]:
from sklearn.experimental import enable_iterative_imputer   # required to unlock IterativeImputer
from sklearn.impute import IterativeImputer

original = np.round(pd.read_csv('50_Startups.csv')[['R&D Spend', 'Administration', 'Marketing Spend']] / 10000)
np.random.seed(9)
original = original.sample(5)
original_with_gaps = original.copy()
original_with_gaps.iloc[1, 0] = np.nan
original_with_gaps.iloc[3, 1] = np.nan
original_with_gaps.iloc[-1, -1] = np.nan

imp = IterativeImputer(estimator=LinearRegression(), max_iter=10, random_state=0)
result = imp.fit_transform(original_with_gaps)
pd.DataFrame(result, columns=original.columns, index=original.index)


Compare this against `df3` from the manual 3-round process above — they should be close (not necessarily identical, since `IterativeImputer` runs more rounds by default and handles convergence checking automatically).

## Summary

| Task | Manual version (above) | sklearn version |
|---|---|---|
| Starting guess | mean-fill every missing value | same, internally |
| Re-predict a column from the others | fit `LinearRegression`, `.predict()` | handled automatically per column |
| Repeat until stable | manually re-run for N iterations | `max_iter=N`, with automatic convergence checking |
| One-line equivalent | — | `IterativeImputer(estimator=LinearRegression(), max_iter=N)` |

### Why go through the manual version at all
`IterativeImputer` can otherwise feel like a black box — seeing each round's regression, prediction, and round-over-round change by hand makes it clear it's just repeated regression, not something more exotic.